<a href="https://colab.research.google.com/github/bookwork123/MachineLearning/blob/main/HW7/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Why is feature scaling essential prior to executing K-Means, DBSCAN, or HDBSCAN on the Wine dataset? Specifically, how would unscaled features distort distance calculations relative to other features? Provide specific examples.**

Feature scaling is essential because K-Means, DBSCAN, and HDBSCAN all group wines by the distance between them, and that distance adds up the differences in every feature in its original units. The Wine features are on very different scales. Proline ranges from 278 to 1,680 with a standard deviation of 314.91, while Hue ranges from 0.48 to 1.71 (standard deviation 0.23) and Nonflavanoid Phenols ranges from 0.13 to 0.66 (standard deviation 0.12).

Without scaling, a difference of 100 in Proline, which is less than a third of its standard deviation, would count for far more than the full range of Hue or Nonflavanoid Phenols. Proline alone makes up 99.77% of the total variance of the unscaled features, and Magnesium (standard deviation 14.28) makes up most of the rest. The distances would essentially only measure Proline, so the clusters would just be bands of Proline values, and features like Flavanoids and Hue would have almost no effect. For DBSCAN and HDBSCAN, eps and the density of each neighborhood would also be measured in Proline units. After StandardScaler, every feature has a standard deviation of 1, so each one has an equal say in the distance.

**2. How did the Elbow Method and Silhouette Analysis in Task 2 reflect the natural structure of the Wine dataset? Did both evaluation metrics align with the true number of wine cultivars (k=3)? Explain any discrepancies observed.**

Both the Elbow Method and Silhouette Analysis suggested k=3, which matches the three wine cultivars. Inertia dropped from 1,658.76 at k=2 to 1,277.93 at k=3, but only to 1,175.43 at k=4, so the curve bends at k=3. The silhouette score was also highest at k=3 (0.2849), ahead of k=4 (0.2602) and k=2 (0.2593).

Although the two metrics agreed, neither one pointed to k=3 very strongly. The elbow is not sharp, since inertia keeps falling steadily after k=3 all the way down to 845.90 at k=10. The silhouette scores are low overall (0.2849 on a scale from -1 to 1), and k=2 and k=4 are only slightly behind k=3, which means the cultivars overlap in the 13-dimensional space instead of forming clearly separated groups.

**3. Compare the clustering behavior and cluster assignments of K-Means against DBSCAN and HDBSCAN on the 13-dimensional Wine feature space. Why did DBSCAN and HDBSCAN classify specific wine samples as noise, while K-Means assigned every sample to a cluster?**

K-Means split the wines into 3 clusters of 65, 51, and 62 wines and assigned every wine to one of them. These clusters lined up closely with the cultivars: one held all 59 Class 0 wines plus 3 Class 1 wines, one held 65 Class 1 wines, and one held all 48 Class 2 wines plus 3 Class 1 wines.

DBSCAN (eps=2.3, min_samples=3) found 3 clusters and labeled 32 wines as noise. Its largest cluster (100 wines) merged 56 Class 0 wines with 44 Class 1 wines, the second held 42 Class 2 wines and 1 Class 1 wine, and the third held only 3 Class 1 wines. HDBSCAN (min_cluster_size=5) found 2 clusters and labeled 58 wines as noise. One cluster again mixed Class 0 (53) and Class 1 (33), and the other was almost all Class 2 (33 of 34). For both, most of the noise came from Class 1 (23 of 32 for DBSCAN and 37 of 58 for HDBSCAN).

K-Means has no concept of noise. It always assigns each wine to the nearest of the k centers, no matter how far away that center is. DBSCAN and HDBSCAN only form clusters in dense regions, so a wine without enough close neighbors is labeled noise. In the PCA plots in Tasks 5 and 6, the noise points are mostly in the sparse area between the dense groups and around the edges, which is where the Class 1 wines that didn't fit a dense group ended up. Density-based clustering also explains why Class 0 and Class 1 were merged: they are connected by a chain of close wines, so DBSCAN and HDBSCAN treat them as one region, while K-Means splits them because it only looks at which center is closest.

**4. Contrast the operational mechanisms of standard DBSCAN and HDBSCAN. How does HDBSCAN's hierarchical, stability-based approach overcome DBSCAN's vulnerability to a single global `eps` parameter when clustering chemical datasets with varying local densities?**

DBSCAN uses one eps for the whole dataset. A wine is a core point if at least min_samples wines are within eps of it, and clusters are made of core points that are connected to each other, plus the border points next to them. Everything else is noise. Because eps is the same everywhere, DBSCAN assumes all clusters have about the same density. Task 5 showed how sensitive this is: eps=2.0 (min_samples=5) gave 5 clusters and 85 noise points, eps=2.3 (min_samples=3) gave 3 clusters and 32 noise points, and eps=2.5 (min_samples=5) merged almost everything into 1 cluster. A small eps breaks sparse groups into noise, while a large eps merges dense groups together.

HDBSCAN doesn't need an eps. It first measures how dense the area around each wine is, then makes wines in sparse areas count as farther apart. It builds a tree from these distances, like the single-linkage dendrogram in Task 7, which covers every possible eps at once. As the distance threshold gets smaller, clusters split apart or lose points, and HDBSCAN ignores any split smaller than min_cluster_size. It then picks the clusters that last over the widest range of thresholds (the most stable ones). Since each cluster can be picked at a different level of the tree, a dense cluster and a sparse cluster can both be found, and I only had to set min_cluster_size=5.

On the Wine data, this still had limits. HDBSCAN found 2 clusters, labeled 58 wines (about a third) as noise, and merged most of Class 0 and Class 1. With only 178 wines in 13 dimensions, the density differences between the cultivars aren't strong enough for either method to separate all three.

**5. Based on your comparative classification experiment in Task 8, how effectively did the cluster assignments from K-Means, DBSCAN, and HDBSCAN enable a downstream classifier to predict the true ground-truth wine cultivars (Class 0-2)? Discuss the limitations of mapping unsupervised clusters to supervised class labels when dealing with noise and density-based groupings.**

The K-Means cluster labels worked very well. The Random Forest reached 0.9722 test accuracy (35 of 36 test wines), because each K-Means cluster mostly matched one cultivar. HDBSCAN reached 0.7500 and DBSCAN reached 0.7222, which is a difference of only one test wine. Both merged most of Class 0 and Class 1 into one cluster, so the classifier could only predict the majority cultivar for that whole cluster and got the rest of it wrong.

There are several limitations to mapping clusters to class labels. A classifier can't do better than the clusters it is given: if two cultivars share a cluster, nothing in that label can tell them apart. The noise label (-1) is treated as one category even though it contains wines from all three cultivars (3, 23, and 6 for DBSCAN, and 6, 37, and 15 for HDBSCAN), so every noise wine gets the same prediction. Very small clusters, like DBSCAN's 3-wine cluster, give the classifier almost nothing to learn from. Density-based clusters are formed by how wines are connected, not by cultivar, so there's no guarantee that a cluster lines up with a class.

The silhouette scores also didn't match the accuracy. HDBSCAN had the highest silhouette score (0.3524), but it was only computed on its 120 non-noise wines, so leaving out the hardest wines made its clusters look better than they are. K-Means had a lower score (0.2849) even though it was computed on all 178 wines and gave the best accuracy. Finally, the test set only has 36 wines, so each wine changes the accuracy by about 2.8%.